# Contrastive Embedding Practice

**Topic:** Contrastive Learning &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(4000, 32)`, `y_train`: 8 classes pushed through a shared nonlinear mixing map, so raw Euclidean neighbors are unreliable.
- `X_test` `(1200, 32)`: classify these. `y_test` is used by the evaluation cell.

## Your task

1. Learn an embedding function from the labeled training data. Options, in increasing coolness: LDA, or a small PyTorch net trained with a triplet or InfoNCE loss (sample anchor/positive/negative from `y_train`).
2. Embed both sets, run k-NN **in the embedding space**, and set `predictions` for `X_test`.
3. Optional and not graded: compare against k-NN on the raw inputs to see what the embedding bought you.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(351)
n_classes, k_lat, d = 8, 6, 32
class_centers = rng.normal(0, 2.4, (n_classes, k_lat))
M1 = rng.normal(0, 1.0, (k_lat, 24))
M2 = rng.normal(0, 1.0, (24, d))

def make(n, seed):
    r = np.random.default_rng(seed)
    y = r.integers(0, n_classes, n)
    z = class_centers[y] + r.normal(0, 1.0, (n, k_lat))
    X = np.tanh(z @ M1) @ M2 + r.normal(0, 0.6, (n, d))
    return X, y

X_train, y_train = make(4000, 361)
X_test, y_test = make(1200, 362)
print(f"X_train {X_train.shape}, {n_classes} classes")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

LDA is the classic 'learn a metric from labels' baseline: it finds directions that pull same-class points together and push classes apart, exactly what a contrastive loss optimizes. k-NN in the LDA embedding then classifies the test set.

```python
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier

lda = LinearDiscriminantAnalysis(n_components=7)
Ztr = lda.fit_transform(X_train, y_train)
Zte = lda.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(Ztr, y_train)
predictions = knn.predict(Zte)
```
